# VoxShield — anti-spoof training on Kaggle

Kaggle instead of Colab. The differences that actually matter:

| | Colab free | Kaggle free |
|---|---|---|
| GPU | T4 16 GB | **P100 16 GB** or **T4 x2** (32 GB total) |
| Quota | opaque, exhausts | **30 GPU-hours/week**, visible |
| Session | ~12 h | **12 h**, and `Save & Run All` runs with the tab closed |
| Storage | Drive, mounted | `/kaggle/working` **20 GB**, persists as notebook output |
| Google Drive | mountable | **not mountable — there is no equivalent** |
| Internet | on | **OFF by default — you must turn it on** |

### Two things to do before running anything

1. **Settings → Internet → On.** Needs phone verification on your account.
   Without it `pip install` and the wav2vec2 download both fail.
2. **Add Data** (top right) → search **`asvpoof-2019-dataset-la`** → Add.
   That is ASVspoof 2019 LA, already on Kaggle — no download, no Drive, it
   mounts read-only at `/kaggle/input/`.

> **On mounting Drive:** you can't. Kaggle has no `google.colab.drive`. `gdown`
> on a share link works for small files but fails on multi-GB ones (Drive's
> virus-scan interstitial and quota errors). Use Kaggle Datasets instead —
> which for ASVspoof means someone has already done the work for you.

> **The DF archives on your Drive:** ASVspoof 2021 is also on Kaggle as
> `mohammedabdeldayem/avsspoof-2021`. Attach that for the cross-condition
> evaluation in section 8 rather than moving 34 GB off Drive.

Nothing below hardcodes a dataset path. Kaggle re-packagers nest and rename
things freely, so the layout is **discovered** — splits are identified from the
utterance ids (`LA_T_`, `LA_D_`, `LA_E_`, `DF_E_`), which are reliable in a way
folder names are not.

## 1 · Environment

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

import os, shutil, torch

print("\ntorch      ", torch.__version__)
print("cuda       ", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        p = torch.cuda.get_device_properties(i)
        print(f"  gpu {i}      {p.name}  {p.total_memory/1024**3:.1f} GB  sm_{p.major}{p.minor}")
    print("bf16       ", torch.cuda.is_bf16_supported())

print("cpus       ", os.cpu_count())
total, used, free = shutil.disk_usage("/kaggle/working")
print(f"disk       {free/1024**3:.0f} GB free")

# Neither a P100 (sm_60) nor a T4 (sm_75) supports bf16, so the trainer's
# resolve_precision() falls back to fp16 + GradScaler on its own.
#
# If you were given T4 x2, this project uses ONE GPU - it does not shard. The
# second card sits idle; that is fine and not worth the complexity here.

In [ ]:
# Internet check. Everything below needs it, and the failure is confusing if
# you only find out three cells later.

# Kaggle's Internet setting is PER NOTEBOOK and defaults to off. It does not
# carry over from another notebook, which is the usual reason this fails.
import socket

socket.setdefaulttimeout(8)

online = True
try:
    socket.socket(socket.AF_INET, socket.SOCK_STREAM).connect(("pypi.org", 443))
except Exception as exc:
    online = False
    detail = f"{type(exc).__name__}: {exc}"

if online:
    print("internet: ON")
else:
    # Raised OUTSIDE the except block on purpose. Raising from inside one
    # chains the exceptions, and IPython's traceback formatter then fails on
    # the chain and buries this message under its own internal errors.
    print("=" * 62)
    print("  INTERNET IS OFF")
    print("=" * 62)
    print(f"  {detail}")
    print()
    print("  Right-hand panel -> Settings -> Internet -> On, then re-run.")
    print("  It needs phone verification on your Kaggle account.")
    print()
    print("  This notebook cannot run without it: speechbrain installs from")
    print("  PyPI and the ECAPA-TDNN weights download from HuggingFace.")
    print("=" * 62)
    raise RuntimeError("Enable Internet in Settings, then re-run this cell.")

## 2 · What did you attach?

This walks `/kaggle/input` and reports every ASVspoof split it can find, by
utterance-id prefix rather than by folder name.

In [ ]:
!ls /kaggle/input/

In [ ]:
import sys, subprocess
from pathlib import Path

import os

# Step out of the repo before deleting it. A previous run of this cell ends with
# `%cd .../voxshield`, so re-running it would otherwise rmtree the kernel's own
# working directory - after which every subprocess fails with
# "fatal: Unable to read current working directory".
os.chdir("/kaggle/working")

WORK = Path("/kaggle/working/voxshield")
if WORK.exists():
    import shutil as _sh; _sh.rmtree(WORK)

subprocess.run(["git", "clone", "-q", "--depth", "1", "-b", "karthik",
                "https://github.com/SathvikGuttula/NullBox.git", str(WORK)], check=True)
sys.path.insert(0, str(WORK / "backend"))

%cd /kaggle/working/voxshield
!git log --oneline -2

In [ ]:
!pip install -q "transformers>=4.44,<5" soundfile

import importlib
for m in ["torch", "torchaudio", "transformers", "soundfile", "librosa", "sklearn"]:
    try:
        print(f"{m:14}", importlib.import_module(m).__version__)
    except Exception as e:
        print(f"{m:14} MISSING  {type(e).__name__}")

In [ ]:
# Scans each attached dataset separately rather than all of /kaggle/input at
# once. On a network mount the cost is per file, so if a 611k-file DF dataset
# is attached you still see the LA result immediately instead of waiting for
# everything.
from pathlib import Path
import time

from app.ml import discovery

INPUT = Path("/kaggle/input")
datasets = sorted(p for p in INPUT.iterdir() if p.is_dir())
print(f"{len(datasets)} dataset(s) attached\n")

layouts = {}
for d in datasets:
    print(f"=== {d.name} ===")
    start = time.perf_counter()
    layout = discovery.discover(d, verbose=True)
    print(f"    ({time.perf_counter() - start:.1f}s)")
    print(layout.describe())
    print()
    if layout.audio:
        layouts[d.name] = layout

if not layouts:
    raise SystemExit(
        "No ASVspoof audio found under /kaggle/input.\n"
        "Use 'Add Data' (top right) and attach 'asvpoof-2019-dataset-la'."
    )

# The training corpus is whichever dataset actually has a train split.
trainable = [n for n, l in layouts.items() if "train" in l.audio]
print(f"usable for training : {trainable or 'NONE'}")
print(f"usable for eval only: {[n for n in layouts if n not in trainable]}")

if not trainable:
    print(
        "\n!! Nothing here can train a model. ASVspoof 2021 DF is evaluation-only.\n"
        "!! Attach 'asvpoof-2019-dataset-la' as well."
    )

for name, layout in layouts.items():
    missing = [s for s in layout.audio if s not in layout.protocols]
    if missing:
        print(f"\n!! {name}: audio but no labels for {missing}")
        print("!! DF ships its keys separately (DF-keys-full.tar.gz).")

## 3 · Manifests

`--discover` uses the layout found above. Paths land as absolute
`/kaggle/input/...`, which is correct — that mount is read-only and outside the
repo.

`train_heldout.csv` / `val_unseen.csv` hold attacks A05–A06 out of training.
Train and dev share A01–A06, so dev EER saturates near zero within a few epochs
while eval-attack performance is still improving; selecting on dev picks a
memorising checkpoint.

In [ ]:
!python backend/scripts/build_manifest.py \
    --discover /kaggle/input \
    --manifest-dir /kaggle/working/voxshield/datasets/manifests \
    --holdout-attacks A05,A06

In [ ]:
!python backend/scripts/dataset_stats.py --all --check-audio --check-limit 1500

## 4 · Waveform cache

Into `/kaggle/temp`, **not** `/kaggle/working`. Working is capped at 20 GB and
persists as the notebook's output; an 8 GB rebuildable cache would eat 40 % of
that quota for nothing. Temp is roomier and disposable — the cache costs a few
minutes to rebuild next session, and the checkpoint is what actually needs to
survive.

In [ ]:
!mkdir -p /kaggle/temp/cache
!python backend/scripts/cache_dataset.py \
    --manifest datasets/manifests/train_heldout.csv \
    --output /kaggle/temp/cache/train_heldout \
    --max-seconds 6.0

!python backend/scripts/cache_dataset.py \
    --manifest datasets/manifests/val_unseen.csv \
    --output /kaggle/temp/cache/val_unseen \
    --max-seconds 6.0

!du -sh /kaggle/temp/cache/* ; df -h /kaggle/temp | tail -1

## 5 · Size the run before committing GPU quota to it

You have 30 GPU-hours a week. `--dry-run` times a dozen real steps and prints
projected per-epoch and total wall clock plus peak VRAM, then stops — so a
misjudged batch size costs thirty seconds, not three hours.

In [ ]:
!python backend/scripts/train_model.py \
    --cache-dir /kaggle/temp/cache \
    --train-manifest datasets/manifests/train_heldout.csv \
    --validation-manifest datasets/manifests/val_unseen.csv \
    --batch-size 32 --gradient-accumulation 1 \
    --epochs 5 --freeze-epochs 1 --unfreeze-top-layers 6 \
    --num-workers 2 \
    --dry-run

## 6 · Train

Checkpoints go to `/kaggle/working`, which persists as this notebook's output.
`--resume` picks up `experiments/kaggle/last.pt` — weights, AdamW moments, LR
schedule and epoch counter — so a 12-hour cutoff costs one epoch.

**Use `Save Version` → `Save & Run All (Commit)`** to run this with the tab
closed. Interactive sessions die when you disconnect; committed ones do not.

In [ ]:
!python backend/scripts/train_model.py \
    --cache-dir /kaggle/temp/cache \
    --train-manifest datasets/manifests/train_heldout.csv \
    --validation-manifest datasets/manifests/val_unseen.csv \
    --batch-size 32 --gradient-accumulation 1 \
    --epochs 5 --freeze-epochs 1 --unfreeze-top-layers 6 \
    --num-workers 2 \
    --experiment-id kaggle \
    --experiments-dir /kaggle/working/experiments \
    --model-out /kaggle/working/models/voxshield_antispoof.pt \
    --resume

In [ ]:
# What survives this session. Anything not under /kaggle/working is gone.
!ls -lh /kaggle/working/models/ /kaggle/working/experiments/kaggle/ 2>/dev/null
!du -sh /kaggle/working

### Continuing in a later session

`/kaggle/working` is wiped when a session ends, but its contents are saved as
the version's **output**. To carry the checkpoint forward:

1. **Save Version** on this notebook.
2. In the new session: **Add Data → Your Work → Notebook Output → this notebook**.
3. It mounts at `/kaggle/input/<notebook-slug>/`. Copy the checkpoint back and
   point `--resume-from` at it:

```python
!mkdir -p /kaggle/working/experiments/kaggle
!cp /kaggle/input/<notebook-slug>/experiments/kaggle/last.pt /kaggle/working/experiments/kaggle/
```

then add `--resume-from /kaggle/working/experiments/kaggle/last.pt` to the
training cell. Raise `--epochs` too, or it will report that the schedule is
already finished and exit.

## 7 · Evaluate on ASVspoof 2019 LA eval

71,237 utterances, attacks **A07–A19 — none seen in training**.

In [ ]:
!python backend/scripts/evaluate_model.py \
    --manifest datasets/manifests/test.csv \
    --model /kaggle/working/models/voxshield_antispoof.pt \
    --calibrate-on datasets/manifests/validation.csv \
    --batch-size 32 --num-workers 2 \
    --save-scores \
    --output-dir /kaggle/working/eval_la_eval

## 8 · Cross-condition evaluation on ASVspoof 2021 DF

The same model against unseen attacks **plus** codec degradation it never
trained on — 611,829 utterances. Expect the EER to be several times worse than
on LA eval. That is the honest result, and showing it beside the LA number is
far more credible than showing only the good one.

**Attach the 34.49 GB DF dataset** (Add Data → search "ASVspoof 2021 DF").
34.49 GB is the right size for the real thing.

Three things decide how this runs, and the cell below reports all three rather
than assuming:

1. **Pre-extracted or still archived?** If the uploader left `.flac` files, they
   mount read-only and cost you no writable disk — evaluate straight off
   `/kaggle/input`. If they left `.tar.gz` parts, each one needs ~10 GB of
   writable space, so they get extracted to `/kaggle/temp` one at a time.
2. **Are the labels included?** DF audio ships *without* them.
   `trial_metadata.txt` comes from `DF-keys-full.tar.gz` (26 MB) — the file you
   already have on Drive. If it is missing, upload just that as a small private
   Kaggle dataset and attach it; nothing else needs to move.
3. **GPU budget.** 611k utterances is roughly 35–70 minutes on a P100, against
   a 30 h/week quota. `--limit` takes a stratified subsample across every attack
   family, so a first pass at 60k costs a few minutes and still covers all of
   them. Say "a 10% stratified subsample" when you report it.

In [ ]:
# Report what is actually attached before doing anything expensive.
from pathlib import Path
import subprocess, collections

from app.ml import discovery

INPUT = Path("/kaggle/input")

archives, flacs, metadata = [], 0, []
for entry in INPUT.rglob("*"):
    if not entry.is_file():
        continue
    name = entry.name.lower()
    if name.endswith((".tar.gz", ".tar", ".zip")) and "df" in str(entry).lower():
        archives.append(entry)
    elif entry.suffix.lower() == ".flac" and entry.stem.startswith("DF_E"):
        flacs += 1
    elif "trial_metadata" in name or "keys" in name:
        metadata.append(entry)

print(f"DF flac files mounted : {flacs:,}")
print(f"DF archives found     : {len(archives)}")
for a in archives:
    print(f"    {a.stat().st_size/1024**3:5.2f} GB  {a}")
print(f"label/metadata files  : {len(metadata)}")
for m in metadata[:5]:
    print(f"    {m}")

layout = discovery.discover(INPUT)
print()
print(layout.describe())

if flacs and not metadata:
    print(
        "\n>> Audio is present but the labels are not. Extract DF-keys-full.tar.gz\n"
        ">> from your Drive, upload trial_metadata.txt as a small Kaggle dataset,\n"
        ">> attach it, and re-run this cell."
    )
elif not flacs and archives:
    print(
        "\n>> The DF data is still archived. Run the extraction cell below; it\n"
        ">> unpacks one part at a time into /kaggle/temp so the 20 GB working\n"
        ">> quota is never touched."
    )
elif flacs and metadata:
    print("\n>> Ready: audio is pre-extracted and labels are present. Skip the\n"
          ">> extraction cell and go straight to evaluation.")

### If the data is still archived — extract, score, delete, repeat

In [ ]:
# Extracts DF one part at a time, scoring and deleting each before the next, so
# peak disk stays near one part rather than the full ~40 GB.
#
# SKIPS ITSELF if DF is not attached. A committed run executes every cell, and
# one raised exception would abort the build and discard the model you just
# spent an hour training.
from pathlib import Path
import subprocess

WORK = Path("/kaggle/working/voxshield")
TEMP = Path("/kaggle/temp/df")
META = next(Path("/kaggle/input").rglob("trial_metadata.txt"), None)

archives = sorted(
    p for p in Path("/kaggle/input").rglob("*")
    if p.name.lower().endswith((".tar.gz", ".tar")) and "df" in str(p).lower()
)

if META is None or not archives:
    print("DF not ready - skipping.")
    print(f"  labels (trial_metadata.txt): {'found' if META else 'MISSING'}")
    print(f"  archives found             : {len(archives)}")
    print("  This is expected while you are only training. Attach the DF")
    print("  dataset and its keys in a later session to run this.")
else:
    TEMP.mkdir(parents=True, exist_ok=True)
    print(f"{len(archives)} archive(s) to process")

    for index, archive in enumerate(archives):
        out_dir = WORK / f"eval_df_part{index}"
        if (out_dir / "scores.csv").exists():
            print(f"part {index} already scored, skipping")
            continue

        print(f"\n=== part {index}: {archive.name} ===")
        subprocess.run(["tar", "xf", str(archive), "-C", str(TEMP)], check=True)

        manifest = WORK / f"datasets/manifests/df_part{index}.csv"
        subprocess.run([
            "python", "backend/scripts/build_manifest.py",
            "--audio-root", str(TEMP), "--protocol", str(META),
            "--split", "test", "--output", str(manifest), "--absolute-paths",
        ], cwd=str(WORK), check=True)

        subprocess.run([
            "python", "backend/scripts/evaluate_model.py",
            "--manifest", str(manifest),
            "--model", "/kaggle/working/models/voxshield_antispoof.pt",
            "--batch-size", "32", "--num-workers", "2",
            "--save-scores", "--no-plots", "--output-dir", str(out_dir),
        ], cwd=str(WORK), check=True)

        subprocess.run(["bash", "-c", f"find {TEMP} -name '*.flac' -delete"], check=False)
        print(f"part {index} done")

### If the data is already extracted — evaluate off the mount directly

In [ ]:
# Pre-extracted case: evaluate straight off the read-only mount.
#
# --limit takes a STRATIFIED subsample across every (label, attack) group, not
# the first N rows, so a subset still covers all attack families. Drop it for
# the full 611k once a subset looks sane.
#
# Also skips itself when DF is absent, for the same reason as the cell above.
from pathlib import Path
import subprocess

df_flacs = any(
    p.stem.startswith("DF_E") for p in Path("/kaggle/input").rglob("*.flac")
)
META = next(Path("/kaggle/input").rglob("trial_metadata.txt"), None)

if not df_flacs or META is None:
    print("DF audio or labels not attached - skipping.")
    print(f"  DF flac present: {df_flacs}   labels present: {META is not None}")
else:
    WORK = "/kaggle/working/voxshield"
    subprocess.run([
        "python", "backend/scripts/build_manifest.py",
        "--discover", "/kaggle/input",
        "--manifest-dir", f"{WORK}/datasets/manifests_df",
        "--holdout-attacks", "", "--absolute-paths",
    ], cwd=WORK, check=True)

    subprocess.run([
        "python", "backend/scripts/evaluate_model.py",
        "--manifest", f"{WORK}/datasets/manifests_df/test.csv",
        "--model", "/kaggle/working/models/voxshield_antispoof.pt",
        "--limit", "60000",
        "--batch-size", "32", "--num-workers", "2",
        "--save-scores", "--output-dir", "/kaggle/working/eval_df",
    ], cwd=WORK, check=True)

If you evaluate DF in pieces (disk, or a session that ran out), pool the
per-utterance scores rather than averaging the per-part EERs — EER is a
property of the whole score distribution, and the average of four is a
different, wrong number:

```
!python backend/scripts/merge_scores.py \
    --scores /kaggle/working/eval_df_part*/scores.csv \
    --output /kaggle/working/eval_df_full --label df
```

## 9 · What to report

Never a single accuracy figure — these corpora are ~90 % spoof, so "always say
spoof" scores 90 % and flags every real customer.

| | LA eval (A07–A19) | DF (pooled) |
|---|---|---|
| EER | | |
| ROC-AUC | | |
| normalised minDCF | | |
| miss rate @ 1 % false alarm | | |

Both `report.json` files contain all four plus a per-attack breakdown.

Two wordings that get checked:

- **normalised minDCF**, not *t-DCF*. Tandem DCF folds in an ASV subsystem's
  scores on the same trials; VoxShield has no enrolled ASV branch yet.
- Always say which split a number came from. "EER 2.1 %" means nothing without
  "on LA eval, attacks unseen in training".

**If LA eval EER comes in under 0.5 %, be suspicious before you are pleased.**
Re-read the `dataset_stats.py` leakage check and confirm you evaluated
`test.csv` and not the split you trained on.